# 04 · Hyperparameter Tuning

**When to run this notebook:** once you are happy with the model set from
notebook 03 and want to optimize the **top** models. You explicitly name which
models to tune — tuning every model on every run is expensive and rarely
worthwhile.

## Approach

`RandomizedSearchCV` (configurable strategy / iterations / folds, ROC-AUC
scoring) searches the named models only; models left out keep their current
snapshot parameters. All four families are still trained and evaluated so the
comparison stays complete — only the **search** is restricted to the named
subset.

> **Sequence:** notebook 4 of 5. See [notebooks.md](./notebooks.md).

## Configuration

- **`IS_DRY_RUN`** (default `True`) — no hyperparameter snapshot, model artifacts,
  or version bump are written.
- **`MODELS_TO_TUNE`** — the explicit subset to search. Accepts friendly names
  (`"xgb"`, `"rf"`, `"lr"`) or class names. Set to e.g. `["xgb"]` to tune only
  XGBoost.

In [ ]:
import sys
import os

# Make the capstone package importable (notebook lives in notebooks/, code in src/capstone/).
sys.path.append(os.path.abspath("../"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory
from pipeline.stages import eda
from pprint import pprint
from utils.tune_hyperparameters import get_all_default_param_grids

IMAGES_FOLDER = "../../../images/results/"
IS_DRY_RUN = True

# Explicit subset of models to search (the "top" models). The rest keep their
# loaded snapshot params. Friendly aliases or class names both work.
MODELS_TO_TUNE = ["xgb"]

# Metric column order used in every results table below. Global (macro) metrics
# come first, then the per-class above/below breakdown.
#   *_macro = sklearn average="macro" = unweighted mean of above- and below-baseline
#             scores. It is the single "overall" number that weights both classes equally.
METRIC_COLS = [
    "roc_auc", "accuracy",
    "precision_macro", "recall_macro", "f1_macro",
    "precision_above", "recall_above", "f1_above",
    "precision_below", "recall_below", "f1_below",
]

### Search grids

Start from the project defaults, then override per model as needed. Only the
grids for models in `MODELS_TO_TUNE` are actually used.

In [2]:
new_grids = get_all_default_param_grids()
pprint(new_grids)

{'LogisticRegression': {'C': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 100.0],
                        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
                        'max_iter': [2000, 5000],
                        'penalty': ['l1', 'elasticnet'],
                        'solver': ['saga']},
 'RandomForestClassifier': {'max_depth': [None, 5, 10, 20, 30],
                            'max_features': ['sqrt', 'log2', None],
                            'min_samples_leaf': [1, 2, 5, 10],
                            'min_samples_split': [2, 5, 10],
                            'n_estimators': [100, 200, 300, 500]},
 'XGBClassifier': {'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
                   'gamma': [0, 0.1, 0.3, 0.5],
                   'learning_rate': [0.01, 0.05, 0.1, 0.2],
                   'max_depth': [3, 4, 5, 6, 8],
                   'min_child_weight': [1, 3, 5, 10],
                   'n_estimators': [100, 200, 300, 500],
                   'subsample': [0.6, 0.7, 0.8, 1.0]}}


### Tuning config

`tune(..., models=MODELS_TO_TUNE)` restricts the search to the named subset.
`snapshot_hyperparams()` and `snapshot_models()` stage the writes (gated by
`IS_DRY_RUN`).

In [ ]:
# snapshot_hyperparams() and snapshot_models() are chained unconditionally — dry_run()
# decides whether they take effect, so no version bump appears during a dry run.
config = (
    VersionConfig.load(use_synthetic=False)
    .tune(strategy="random", n_iter=100, cv=5, new_grids=new_grids, models=MODELS_TO_TUNE)
    .snapshot_hyperparams()
    .snapshot_models()
    .dry_run(IS_DRY_RUN)
    .build()
)

run = PipelineRun(config)
stages = PipelineFactory.tune_hyperparams(config)
print("Scenario:", stages.scenario)
print("Tuning models:", config.tune_model_names)

### Load, process, split & scale

`DataLoader` is the expensive step, so it is run on its own; the rest of the
pre-modeling stages follow.

In [4]:
stages.loader.run(run)
run.summary()

Loaded snapshot 'v3.4_real': 60696 rows from 2026-04-29
  Polls: {'upload': 21398, '24h': 20906, '7d': 18392}
Loaded baselines 'v4.0': 28814 baseline videos, 974 median rows (974 channels)
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['tune']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       None
  df_engineered  None
  df_train       None
  df_test        None
  df_val         None
  df_gen         None
  X_train        None
  X_test         None
  X_val          None
  X_val_unscaled  None
  X_gen          None
  y_train        None
  y_test         None
  y_val          None
  y_gen          None
  models         empty dict
  results        empty dict


In [5]:
stages.preprocessor.run(run)
stages.engineer.run(run)
stages.splitter.run(run)
stages.scaler.run(run)
run.summary()

Building clean dataset
snapshot cols: Index(['video_id', 'poll_timestamp', 'channel_id', 'channel_handle', 'title',
       'view_count', 'like_count', 'comment_count', 'face_count', 'brightness',
       'colorfulness', 'vertical', 'tier', 'description', 'tags',
       'duration_seconds', 'category_id', 'category_name', 'published_at',
       'poll_label', 'hours_since_publish', 'subscriber_count',
       'contains_synthetic_media'],
      dtype='object')

[1/3] Pivoting snapshots...
  Videos with all 3 polls: 18334 (dropped 3111 incomplete)
  Pivoted shape: (18403, 34)

[2/3] Joining baseline medians...
  Baseline join: 18403/18403 videos matched a channel median

[3/3] Cleaning data...
  Dropped 15 row(s) with null contains_synthetic_media (private/failed harvest).
  Cleaned: 18388 rows × 40 columns

Clean dataset: 18388 rows × 40 columns
  all: dropped 402 rows with NaN in a baseline_median_* column or 0.0 baseline_median_engagement_rate
Engineering features

[1/10] Computing target 

### Search + train

With tuning enabled, `ModelTrainer` runs `RandomizedSearchCV` (5-fold CV) on the
named models, then fits all four final models. Models outside `MODELS_TO_TUNE`
are fit with their loaded snapshot params.

In [6]:
stages.trainer.run(run)
if run.tune_elapsed_s is not None:
    mins, secs = divmod(run.tune_elapsed_s, 60)
    print(f"\nTuning wall time: {run.tune_elapsed_s:.1f}s ({int(mins)}m {secs:.0f}s)")
run.summary()

Loaded hyperparams 'v1.1' (saved 2026-04-29)
  Models: ['LogisticRegression', 'RandomForest', 'XGBoost', 'VotingClassifier']
  Search: {'strategy': 'random', 'n_iter': 100, 'cv': 5, 'scoring': 'roc_auc'}
Loaded hyperparams from snapshot 'v1.1'.
  Note: injected l1_ratio=0.5 for elasticnet LR (missing from snapshot).
Tuning subset: ['XGBoost'] (keeping loaded params for ['LogisticRegression', 'RandomForest'])

Using param_grid: {'n_estimators': [100, 200, 300, 500], 'max_depth': [3, 4, 5, 6, 8], 'learning_rate': [0.01, 0.05, 0.1, 0.2], 'subsample': [0.6, 0.7, 0.8, 1.0], 'colsample_bytree': [0.6, 0.7, 0.8, 1.0], 'min_child_weight': [1, 3, 5, 10], 'gamma': [0, 0.1, 0.3, 0.5]}
Tuning XGBClassifier with strategy='random' (cv=5, scoring='roc_auc')
  Sampling 100 combinations from grid of ~20,480 total
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Best roc_auc: 0.8999
Best params:   {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 8, 'learning_rate

### Persist tuned hyperparameters + models

Gated by `IS_DRY_RUN`. With `IS_DRY_RUN=False` these write the new hyperparameter
version and the retrained model artifacts to GCS.

In [7]:
stages.hyperparam_snapshotter.run(run)
stages.model_snapshotter.run(run)

[dry_run] HyperparamSnapshotter — GCS write skipped.
[dry_run] ModelSnapshotter — GCS write skipped.


PipelineRun(model_version='v5.3', num_synth_rows=0, populated=['df_videos', 'df_baselines', 'df_medians', 'df_clean', 'df_engineered', 'df_train', 'df_test', 'df_val', 'df_gen', 'X_train', 'X_test', 'X_val', 'X_val_unscaled', 'X_gen', 'y_train', 'y_test', 'y_val', 'y_gen', 'models'])

### Validate on the locked holdout

In [8]:
stages.validator.run(run)
stages.validation_results_snapshotter.run(run)


=== Validator — validation-set results ===
  lr_l1           AUC=0.7678  acc=0.6997  F1↑=0.7321
  rf              AUC=0.8716  acc=0.7947  F1↑=0.8223
  xgb             AUC=0.9068  acc=0.8268  F1↑=0.8457
  ensemble        AUC=0.9039  acc=0.8270  F1↑=0.8465
[dry_run] ValidationResultsSnapshotter — GCS append skipped.


PipelineRun(model_version='v5.3', num_synth_rows=0, populated=['df_videos', 'df_baselines', 'df_medians', 'df_clean', 'df_engineered', 'df_train', 'df_test', 'df_val', 'df_gen', 'X_train', 'X_test', 'X_val', 'X_val_unscaled', 'X_gen', 'y_train', 'y_test', 'y_val', 'y_gen', 'models', 'results'])

### A note on the evaluation metrics

Every metrics table in this notebook leads with three **global** scores —
`precision_macro`, `recall_macro`, `f1_macro` — followed by the per-class
breakdown (`*_above`, `*_below`).

The global scores use **sklearn's `average="macro"`**: compute precision /
recall / F1 separately for each class, then take the unweighted mean across
classes. For example, `f1_macro = mean(f1_above, f1_below)`. Because the
target is ~50/50 by design, macro and weighted averages are nearly identical
here — macro is preferred because it treats both classes equally regardless
of exact counts, which matches the intent of a balanced binary target.

The per-class split is retained because the error costs differ — a false
"will overperform" prediction has a different business impact than a false
"will underperform" — but the macro scores are shown first as the headline
numbers.

In [9]:
df_val = pd.DataFrame(run.results).T[METRIC_COLS].astype(float).round(4)
df_val.index.name = "model"
df_val.style.highlight_max(color="#d4edda").format("{:.4f}")

,roc_auc,accuracy,precision_macro,recall_macro,f1_macro,precision_above,recall_above,f1_above,precision_below,recall_below,f1_below
model,,,,,,,,,,,
lr_l1,0.7678,0.6997,0.6958,0.6949,0.6953,0.7257,0.7386,0.7321,0.6659,0.6512,0.6585
rf,0.8716,0.7947,0.7954,0.7871,0.7896,0.7919,0.8551,0.8223,0.7988,0.7192,0.7570
xgb,0.9068,0.8268,0.8252,0.8235,0.8242,0.8375,0.8540,0.8457,0.8130,0.7929,0.8028
ensemble,0.9039,0.8270,0.8258,0.8231,0.8242,0.8346,0.8589,0.8465,0.8170,0.7873,0.8019


### Best parameters found

The search results for the tuned models. Compare these against the prior
snapshot to confirm the search moved in a sensible direction (e.g. more
estimators, stronger regularization) and that validation AUC improved or held.

In [10]:
for name, entry in run.models.items():
    cfg = entry["model_config"]
    print(f"{name}  ({cfg.model_type}):")
    pprint(cfg.hyperparameters)
    print()

lr_l1  (LogisticRegression):
{'C': 10.0, 'max_iter': 5000, 'penalty': 'elasticnet', 'solver': 'saga'}

rf  (RandomForest):
{'max_depth': None,
 'max_features': 'sqrt',
 'min_samples_leaf': 2,
 'min_samples_split': 2,
 'n_estimators': 800}

xgb  (XGBoost):
{'colsample_bytree': 1.0,
 'learning_rate': 0.05,
 'max_depth': 8,
 'min_child_weight': 5,
 'n_estimators': 500,
 'subsample': 0.8}

ensemble  (VotingClassifier):
{'estimators': ['rf', 'xgb'], 'voting': 'soft', 'weights': [1, 2]}



## Findings — tuning

<TODO(jelani): Add findings, touching on topics below>

- which models improved on validation ROC-AUC and by how much
- whether the gains justify the added complexity. 
- Based on prior modeling runs, Tree-model gains here are often incremental; the LR ceiling is structural (see notebook 03).

**Next:** [05 · Final model selection + results](./05_final_model_selection.ipynb).

## Persist version bump (GCS)

Runs only when `IS_DRY_RUN=False`.

In [11]:
config.commit()

[dry_run] commit skipped — versions.json not written to GCS.


Exception ignored in: <function ResourceTracker.__del__ at 0x107551300>
Traceback (most recent call last):
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x107591300>
Traceback (most recent call last):
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
Chi